# 📈 Phase 1: Time-Series Fraud Trend Analysis & Forecasting
### **SecureBank AI — Hybrid Fraud Intelligence System**

#### **Module Objectives:**
1. **Temporal Transformation**: Convert raw transaction `Time` (seconds from initial transaction) into elapsed hours (0–47) and 24-hour diurnal cycle time.
2. **Hourly Aggregation**: Group 284,807 transactions into 48 continuous hourly windows to compute transaction volume, fraud frequency, fraud monetary volume ($), and fraud rate percentage (%).
3. **Diurnal Pattern Discovery**: Analyze day vs night fraud velocity and identify vulnerable banking hours.
4. **Rolling Moving Averages**: Compute 3-hour and 6-hour moving averages to detect fraud velocity surges.
5. **Baseline Trend Forecasting**: Train Holt's Exponential Smoothing / Autoregressive trend model to forecast future fraud spikes.
6. **Export Artifacts**: Save `hourly_fraud_trends.csv` and visualization charts for seamless integration into the Streamlit multi-page dashboard.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Styling configurations
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
print("Environment initialized and directories ready.")

## 1. Load Raw Transaction Dataset
We inspect the raw `Time`, `Amount`, and `Class` columns from `Dataset/creditcard.csv`.

In [ ]:
data_path = "../Dataset/creditcard.csv"
if not os.path.exists(data_path):
    data_path = "Dataset/creditcard.csv"

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path, usecols=["Time", "Amount", "Class"])

print(f"Total Transactions: {len(df):,}")
print(f"Total Fraudulent Cases: {df['Class'].sum():,} ({df['Class'].mean()*100:.3f}%)")
print(f"Time Span: {df['Time'].min():.0f}s to {df['Time'].max():.0f}s (~{df['Time'].max()/3600:.1f} hours / 2 days)")
df.head()

## 2. Feature Engineering: Time Windows and Diurnal Cycles
We convert `Time` (seconds) into:
- `Hour_Elapsed`: Total continuous hours (0 through 47)
- `Day`: Day 1 (Hours 0-23) vs Day 2 (Hours 24-47)
- `Hour_Of_Day`: Modulo 24 hour indicator (0 to 23) to analyze peak risk hours

In [ ]:
df["Hour_Elapsed"] = (df["Time"] // 3600).astype(int)
df["Day"] = (df["Hour_Elapsed"] // 24) + 1
df["Hour_Of_Day"] = df["Hour_Elapsed"] % 24

print("Sample of time-engineered records:")
df[["Time", "Hour_Elapsed", "Day", "Hour_Of_Day", "Amount", "Class"]].head()

## 3. Hourly Aggregation & Trend Dataset Construction
We aggregate transactions into hourly time buckets to compute:
- `total_tx`: Total transaction volume per hour
- `fraud_tx`: Number of fraudulent transactions
- `legit_tx`: Number of legitimate transactions
- `fraud_rate_pct`: Percentage of transactions flagged as fraud
- `total_amount`: Total financial volume processed ($)
- `fraud_amount`: Total fraudulent financial loss ($)

In [ ]:
hourly_df = df.groupby("Hour_Elapsed").agg(
    total_tx=("Class", "count"),
    fraud_tx=("Class", "sum"),
    total_amount=("Amount", "sum"),
    fraud_amount=("Amount", lambda x: x[df.loc[x.index, "Class"] == 1].sum())
).reset_index()

hourly_df["legit_tx"] = hourly_df["total_tx"] - hourly_df["fraud_tx"]
hourly_df["fraud_rate_pct"] = (hourly_df["fraud_tx"] / hourly_df["total_tx"]) * 100
hourly_df["Day"] = (hourly_df["Hour_Elapsed"] // 24) + 1
hourly_df["Hour_Of_Day"] = hourly_df["Hour_Elapsed"] % 24

print(f"Aggregated into {len(hourly_df)} hourly bins.")
hourly_df.head(10)

## 4. Visualizing Hourly Transaction Volume vs Fraud Spikes
We plot the dual dynamics of overall banking transactions vs fraudulent incidents across the 48-hour timeline.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Subplot 1: Total Volume vs Fraud Count
color_vol = '#2563eb'
color_fraud = '#dc2626'

ax1.set_title("Hourly Transaction Volume vs. Detected Fraud Count", fontsize=14, fontweight='bold', pad=12)
line1 = ax1.plot(hourly_df["Hour_Elapsed"], hourly_df["total_tx"], color=color_vol, marker='o', linewidth=2, label='Total Transactions')
ax1.set_ylabel("Total Transactions", color=color_vol, fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color_vol)

ax1_twin = ax1.twinx()
line2 = ax1_twin.plot(hourly_df["Hour_Elapsed"], hourly_df["fraud_tx"], color=color_fraud, marker='s', linewidth=2.5, linestyle='--', label='Fraud Count')
ax1_twin.set_ylabel("Fraud Count", color=color_fraud, fontsize=12, fontweight='bold')
ax1_twin.tick_params(axis='y', labelcolor=color_fraud)
ax1.grid(True, linestyle=':', alpha=0.6)

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

# Subplot 2: Fraud Rate Percentage (%)
ax2.set_title("Hourly Fraud Rate Percentage (% of Transactions)", fontsize=14, fontweight='bold', pad=12)
bars = ax2.bar(hourly_df["Hour_Elapsed"], hourly_df["fraud_rate_pct"], color='#f59e0b', alpha=0.85, edgecolor='#b45309')
ax2.axhline(hourly_df["fraud_rate_pct"].mean(), color='#dc2626', linestyle='--', linewidth=1.5, label=f'Mean Fraud Rate ({hourly_df["fraud_rate_pct"].mean():.2f}%)')
ax2.set_xlabel("Elapsed Time (Hours)", fontsize=12, fontweight='bold')
ax2.set_ylabel("Fraud Rate (%)", fontsize=12, fontweight='bold')
ax2.set_xticks(range(0, 48, 2))
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig("../outputs/figures/hourly_fraud_trends.png", dpi=300)
plt.show()
print("Hourly trends figure saved to outputs/figures/hourly_fraud_trends.png")

## 5. Rolling Moving Averages (Smoothing & Spike Detection)
Real-world security operations utilize moving averages to dampen minute-by-minute noise and highlight sustained fraud velocity bursts.

In [ ]:
hourly_df["rolling_fraud_3h"] = hourly_df["fraud_tx"].rolling(window=3, min_periods=1).mean()
hourly_df["rolling_fraud_6h"] = hourly_df["fraud_tx"].rolling(window=6, min_periods=1).mean()
hourly_df["rolling_rate_3h"] = hourly_df["fraud_rate_pct"].rolling(window=3, min_periods=1).mean()

# Fraud velocity spike threshold (mean + 1.5 * std)
spike_thresh = hourly_df["fraud_tx"].mean() + 1.5 * hourly_df["fraud_tx"].std()
hourly_df["is_fraud_spike"] = hourly_df["fraud_tx"] > spike_thresh

plt.figure(figsize=(15, 6))
plt.plot(hourly_df["Hour_Elapsed"], hourly_df["fraud_tx"], color='#94a3b8', alpha=0.7, label='Raw Hourly Fraud Count')
plt.plot(hourly_df["Hour_Elapsed"], hourly_df["rolling_fraud_3h"], color='#3b82f6', linewidth=2.5, label='3-Hour Moving Average')
plt.plot(hourly_df["Hour_Elapsed"], hourly_df["rolling_fraud_6h"], color='#8b5cf6', linewidth=2.5, linestyle='--', label='6-Hour Moving Average')

# Highlight spike points
spikes = hourly_df[hourly_df["is_fraud_spike"]]
plt.scatter(spikes["Hour_Elapsed"], spikes["fraud_tx"], color='#ef4444', s=120, zorder=5, label=f'High-Risk Velocity Spike (> {spike_thresh:.1f} tx/hr)')

plt.title("Fraud Velocity Moving Averages & High-Risk Spike Outliers", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Elapsed Time (Hours)", fontsize=12)
plt.ylabel("Fraud Transaction Count", fontsize=12)
plt.xticks(range(0, 48, 2))
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig("../outputs/figures/fraud_moving_averages.png", dpi=300)
plt.show()

## 6. Diurnal Pattern: Day 1 vs. Day 2 Comparison
We analyze how fraud behaves across the 24 hours of the day. In banking fraud, overnight periods often suffer from disproportionately high fraud rates because legitimate customer transactions drop while automated fraud bots continue.

In [ ]:
diurnal_df = hourly_df.pivot(index="Hour_Of_Day", columns="Day", values=["total_tx", "fraud_tx", "fraud_rate_pct"])

fig, ax = plt.subplots(figsize=(14, 6))
width = 0.35
x = np.arange(24)

ax.bar(x - width/2, diurnal_df[("fraud_rate_pct", 1)], width, label='Day 1 Fraud Rate (%)', color='#0284c7', alpha=0.85)
ax.bar(x + width/2, diurnal_df[("fraud_rate_pct", 2)], width, label='Day 2 Fraud Rate (%)', color='#f97316', alpha=0.85)

ax.set_title("Diurnal Fraud Rate Comparison by Hour of Day (Day 1 vs Day 2)", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("Hour of Day (0:00 to 23:00)", fontsize=12)
ax.set_ylabel("Fraud Rate (%)", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([f"{h:02d}:00" for h in x], rotation=45)
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/figures/diurnal_fraud_comparison.png", dpi=300)
plt.show()

## 7. Short-Term Time-Series Trend Forecasting
We implement **Holt's Linear Exponential Smoothing** model to forecast the final 12 hours of fraud transactions based on the preceding 36 hours of historical data.

**Holt's Model Formulation**:
$$\text{Level: } L_t = \alpha Y_t + (1 - \alpha)(L_{t-1} + T_{t-1})$$
$$\text{Trend: } T_t = \beta (L_t - L_{t-1}) + (1 - \beta) T_{t-1}$$
$$\text{Forecast: } \hat{Y}_{t+h} = L_t + h \cdot T_t$$

We tune smoothing parameters $\alpha$ and $\beta$ via Mean Squared Error optimization.

In [ ]:
class HoltsLinearTrendModel:
    def __init__(self, alpha=0.5, beta=0.2):
        self.alpha = alpha
        self.beta = beta
        self.level = None
        self.trend = None
        
    def fit(self, series):
        y = np.array(series)
        n = len(y)
        
        def loss_func(params):
            a, b = params
            L = np.zeros(n)
            T = np.zeros(n)
            L[0] = y[0]
            T[0] = y[1] - y[0] if n > 1 else 0
            pred = np.zeros(n)
            for t in range(1, n):
                pred[t] = L[t-1] + T[t-1]
                L[t] = a * y[t] + (1 - a) * (L[t-1] + T[t-1])
                T[t] = b * (L[t] - L[t-1]) + (1 - b) * T[t-1]
            return np.mean((y[1:] - pred[1:]) ** 2)
            
        res = minimize(loss_func, [self.alpha, self.beta], bounds=[(0.01, 0.99), (0.01, 0.99)])
        self.alpha, self.beta = res.x
        
        # Final filter run
        self.L_hist = np.zeros(n)
        self.T_hist = np.zeros(n)
        self.L_hist[0] = y[0]
        self.T_hist[0] = y[1] - y[0] if n > 1 else 0
        for t in range(1, n):
            self.L_hist[t] = self.alpha * y[t] + (1 - self.alpha) * (self.L_hist[t-1] + self.T_hist[t-1])
            self.T_hist[t] = self.beta * (self.L_hist[t] - self.L_hist[t-1]) + (1 - self.beta) * self.T_hist[t-1]
            
        self.level = self.L_hist[-1]
        self.trend = self.T_hist[-1]
        return self
        
    def forecast(self, steps):
        h = np.arange(1, steps + 1)
        preds = self.level + h * self.trend
        return np.maximum(0, preds)  # Fraud count cannot be negative

# Train/Test Split: First 36 hours for training, last 12 hours for testing
train_hours = 36
y_train = hourly_df.loc[:train_hours-1, "fraud_tx"].values
y_test = hourly_df.loc[train_hours:, "fraud_tx"].values
test_indices = hourly_df.loc[train_hours:, "Hour_Elapsed"].values

model = HoltsLinearTrendModel().fit(y_train)
y_pred = model.forecast(len(y_test))

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Optimized Alpha: {model.alpha:.3f}, Optimized Beta: {model.beta:.3f}")
print(f"Test Evaluation on 12-Hour Forecast:")
print(f"MAE: {mae:.2f} transactions/hr")
print(f"RMSE: {rmse:.2f} transactions/hr")

# Plot Actual vs Forecast
plt.figure(figsize=(15, 6))
plt.plot(hourly_df["Hour_Elapsed"], hourly_df["fraud_tx"], color='#3b82f6', marker='o', label='Actual Fraud Count (History)')
plt.plot(test_indices, y_pred, color='#ef4444', marker='s', linewidth=2.5, linestyle='--', label=f'Holt Trend Forecast (MAE: {mae:.2f})')
plt.fill_between(test_indices, np.maximum(0, y_pred - 1.96*rmse), y_pred + 1.96*rmse, color='#ef4444', alpha=0.15, label='95% Prediction Confidence Band')
plt.axvline(train_hours - 0.5, color='#64748b', linestyle=':', label='Forecast Horizon Boundary')

plt.title("Short-Term Fraud Volume Trend Forecasting (Actual vs Forecast)", fontsize=14, fontweight='bold', pad=12)
plt.xlabel("Elapsed Time (Hours)", fontsize=12)
plt.ylabel("Fraud Transaction Count", fontsize=12)
plt.xticks(range(0, 48, 2))
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig("../outputs/figures/time_series_forecast.png", dpi=300)
plt.show()

## 8. Export Processed Time-Series Data for Dashboard Integration
We save `hourly_fraud_trends.csv` so the Streamlit dashboard (`Fraud Trends` tab) can load the data instantly without expensive recomputation.

In [ ]:
output_csv = "../data/processed/hourly_fraud_trends.csv"
hourly_df.to_csv(output_csv, index=False)
print(f"Successfully saved hourly fraud trends to: {output_csv}")
print(f"Exported Rows: {len(hourly_df)}, Columns: {list(hourly_df.columns)}")
print("\nPhase 1 Time Series Analysis completed successfully! Ready for Phase 2: Isolation Forest.")